In [1]:
import cv2
import os
import glob
import numpy as np
from collections import defaultdict

# 1. Configuración de rutas
# Asegúrate de que esta imagen exista en esa ruta
query_image_path = "../data/test_images/my_messy_500_peso_photo.jpg" 
database_dir = "../data/database/orb_database/"

# 2. Inicializar ORB y el Matcher
orb = cv2.ORB_create(nfeatures=2000)
# NORM_HAMMING es obligatorio para comparar descriptores de ORB
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False) 

def recognize_banknote(query_path, db_dir):
    print(f"--- Analizando Imagen de Consulta: {os.path.basename(query_path)} ---")
    
    query_img = cv2.imread(query_path)
    if query_img is None:
        print("Error: No se pudo cargar la imagen de consulta. Verifica la ruta.")
        return
        
    gray_query = cv2.cvtColor(query_img, cv2.COLOR_BGR2GRAY)
    kp_query, des_query = orb.detectAndCompute(gray_query, None)
    
    if des_query is None:
        print("No se encontraron características en la imagen de consulta.")
        return

    category_votes = defaultdict(int)
    db_files = glob.glob(os.path.join(db_dir, "*.npy"))
    
    if not db_files:
        print("Error: No se encontraron archivos .npy en la base de datos de ORB. Corre el archivo Regions primero.")
        return
    
    for db_file in db_files:
        des_db = np.load(db_file)
        
        # Ignorar si el componente no tiene descriptores guardados
        if des_db is None or len(des_db) == 0:
            continue
            
        filename = os.path.basename(db_file)
        # Se asume la nomenclatura del preprocesamiento: "norm_clean_500PesosFront_comp_0.npy"
        category = filename.split("_comp_")[0].replace("norm_clean_", "")
        
        # 3. K-Nearest Neighbors (K=2 para el ratio test)
        matches = matcher.knnMatch(des_query, des_db, k=2)
        
        # 4. Ratio Test de Lowe adaptado
        good_matches = 0
        for match_pair in matches:
            if len(match_pair) == 2:
                m, n = match_pair
                # Si la mejor coincidencia es significativamente mejor que la segunda, se acepta
                if m.distance < 0.75 * n.distance:
                    good_matches += 1
                    
        # 5. Conteo de votos (solo si hay más de 5 coincidencias sólidas)
        if good_matches > 5:
            category_votes[category] += good_matches
            print(f"  {good_matches} coincidencias con el componente {category}.")

    # 6. Resultados finales
    if not category_votes:
        print("\nRESULTADO: No se reconoció el billete. Coincidencias por debajo del umbral.")
    else:
        sorted_results = sorted(category_votes.items(), key=lambda item: item[1], reverse=True)
        winner, top_score = sorted_results[0]
        
        print("\n" + "="*40)
        print(f"BILLETE PREDICHO: {winner}")
        print(f"   Total de Características Coincidentes: {top_score}")
        print("="*40)
        
        if len(sorted_results) > 1:
            print(f"Segundo lugar: {sorted_results[1][0]} ({sorted_results[1][1]} coincidencias)")

# Ejecutar la función principal
recognize_banknote(query_image_path, database_dir)

--- Analizando Imagen de Consulta: my_messy_500_peso_photo.jpg ---
  7 coincidencias con el componente 1000PesosBack.
  31 coincidencias con el componente 1000PesosFront.
  9 coincidencias con el componente 1000PesosFront.
  56 coincidencias con el componente 100PesosBack.
  15 coincidencias con el componente 100PesosBack.
  59 coincidencias con el componente 100PesosFront.
  7 coincidencias con el componente 100PesosFront.
  21 coincidencias con el componente 200PesosBack.
  10 coincidencias con el componente 200PesosBack.
  16 coincidencias con el componente 200PesosFront.
  26 coincidencias con el componente 20PesosBack.
  6 coincidencias con el componente 20PesosBack.
  28 coincidencias con el componente 20PesosFront.
  41 coincidencias con el componente 20PesosPolimeroBack.
  8 coincidencias con el componente 20PesosPolimeroBack.
  8 coincidencias con el componente 20PesosPolimeroFront.
  11 coincidencias con el componente 20PesosPolimeroFront.
  17 coincidencias con el componente